# 01 数据结构：Series 与 DataFrame

pandas 只有两个核心对象：

| 对象 | 类比 | 关键特征 |
| --- | --- | --- |
| `Series` | 带标签的一维数组（Excel 的一列） | 一组值 + 一个索引 |
| `DataFrame` | 二维表（Excel 的一张表） | 行索引 + 列名 + 每列一个 dtype |

学会 pandas 处理数据，本质是学会三件事：**怎么造表、怎么取表、怎么改表**。
本节先把"表"本身认清楚，后面所有章节都建立在这两个对象上。

> 运行环境：`llm_sft`（Python 3.11 / pandas 3.0.5）。本 notebook 不依赖任何数据文件，可以直接跑。

## 0. 环境准备

In [1]:
import numpy as np
import pandas as pd

# 让宽表在 notebook 里别被省略号截断
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
pd.set_option("display.unicode.east_asian_width", True)  # 中文对齐

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 3.0.5 | numpy 2.4.6


## 1. Series：值 + 索引

构造 Series 的三种常见方式：列表、字典、标量广播。

In [2]:
s = pd.Series([10, 20, 30])
print("值      :", s.tolist())
print("索引    :", s.index.tolist())
print("dtype   :", s.dtype)
print("name    :", s.name)

s

值      : [10, 20, 30]
索引    : [0, 1, 2]
dtype   : int64
name    : None


0    10
1    20
2    30
dtype: int64

In [3]:
# 用字典构造：字典的 key 自动变成索引
cities = pd.Series({"北京": 2189, "上海": 2487, "广州": 1873})
print(cities)
print("---")
# 标量广播：同一个值填满所有位置
print(pd.Series(0.0, index=["a", "b", "c"], name="zero"))

北京    2189
上海    2487
广州    1873
dtype: int64
---
a    0.0
b    0.0
c    0.0
Name: zero, dtype: float64


### Series 的运算会**按索引对齐**

这是 pandas 最容易被误解、也最强大的行为：两个 Series 相加时，先对齐索引，再计算；
对不上的位置结果是 `NaN`（缺失），而不是报错。

> 要点：pandas 的默认是"标签对齐"，不是"按位置对齐"。想按位置对齐用 `.values` 或 `df.iloc`。

In [4]:
s1 = pd.Series([1, 2, 3], index=["a", "b", "c"])
s2 = pd.Series([10, 20], index=["b", "d"])
print("s1 + s2 =")
print(s1 + s2)  # a、c 只有一边有 → NaN；b 取 2+10；d 取 20

s1 + s2 =
a     NaN
b    12.0
c     NaN
d     NaN
dtype: float64


In [5]:
# 配套的取值方式
print("s1['b']       :", s1["b"])          # 按标签
print("s1.loc['b']   :", s1.loc["b"])      # 显式按标签
print("s1.iloc[1]    :", s1.iloc[1])       # 按位置：第 2 个

# pandas 3.0 起，整数下标一律按"标签"解释（不再按位置），所以这里会 KeyError
try:
    s1[1]
except KeyError as e:
    print("s1[1] 报错 KeyError:", e)
    print("→ 要按位置取，用 .iloc[1]")

s1['b']       : 2
s1.loc['b']   : 2
s1.iloc[1]    : 2
s1[1] 报错 KeyError: 1
→ 要按位置取，用 .iloc[1]


## 2. DataFrame：一张表

四种最常用的构造方式，注意它们对"缺字段"的默认处理不同。

In [6]:
# 方式一：字典的列表（每列一个 list，最常用）
df = pd.DataFrame({
    "name": ["张三", "李四", "王五"],
    "age": [23, 31, 27],
    "city": ["北京", "上海", "广州"],
})
df

,name,age,city
0,张三,23,北京
1,李四,31,上海
2,王五,27,广州


In [7]:
# 方式二：字典的列表（每行一个 dict）——缺的字段自动变 NaN
records = [
    {"name": "张三", "age": 23, "city": "北京"},
    {"name": "李四", "age": 31},              # 少 city
    {"name": "王五", "age": 27, "city": "广州", "vip": True},
]
pd.DataFrame(records)

,name,age,city,vip
0,张三,23,北京,NaN
1,李四,31,NaN,NaN
2,王五,27,广州,True


In [8]:
# 方式三：字典的 Series（按索引对齐，对不上的补 NaN）
pd.DataFrame({
    "a": pd.Series([1, 2], index=["x", "y"]),
    "b": pd.Series([3, 4, 5], index=["y", "z", "x"]),
})

,a,b
x,1.0,5
y,2.0,3
z,NaN,4


In [9]:
# 方式四：numpy 数组 + 显式指定列名和行索引
arr = np.arange(6).reshape(2, 3)
pd.DataFrame(arr, index=["r1", "r2"], columns=["c1", "c2", "c3"])

,c1,c2,c3
r1,0,1,2
r2,3,4,5


## 3. 认清三件套：行索引、列名、dtype

In [10]:
df = pd.DataFrame(
    {
        "city": ["北京", "上海", "上海", "广州"],
        "month": ["2024-01", "2024-01", "2024-02", "2024-02"],
        "sales": [120.5, 88.0, 95.5, 60.0],
    },
    index=["r1", "r2", "r3", "r4"],
)

print("形状 shape   :", df.shape)          # (行数, 列数)
print("行索引 index :", df.index.tolist())
print("列名 columns :", df.columns.tolist())
print("每列类型      :", df.dtypes.to_dict())

形状 shape   : (4, 3)
行索引 index : ['r1', 'r2', 'r3', 'r4']
列名 columns : ['city', 'month', 'sales']
每列类型      : {'city': <StringDtype(na_value=nan)>, 'month': <StringDtype(na_value=nan)>, 'sales': dtype('float64')}


In [11]:
df.info()   # 一眼看行数、列名、非空数量、内存占用——拿到新数据第一步就该跑它

<class 'pandas.DataFrame'>
Index: 4 entries, r1 to r4
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   city    4 non-null      str    
 1   month   4 non-null      str    
 2   sales   4 non-null      float64
dtypes: float64(1), str(2)
memory usage: 188.0 bytes


In [12]:
# 转置：行变列、列变行（注意转置后所有列都变成同一种 dtype）
df.T

,r1,r2,r3,r4
city,北京,上海,上海,广州
month,2024-01,2024-01,2024-02,2024-02
sales,120.5,88.0,95.5,60.0


## 4. 取列、加列、删列

- `df["列名"]` → Series；`df[["列1", "列2"]]` → DataFrame（双层方括号）
- `df.列名` 也能取，但**不要依赖**：列名和 DataFrame 的方法同名时会失效

In [13]:
print(type(df["sales"]))
print(type(df[["city", "sales"]]))
df[["city", "sales"]]

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,city,sales
r1,北京,120.5
r2,上海,88.0
r3,上海,95.5
r4,广州,60.0


In [14]:
# 属性访问的陷阱：count 是列名，但 df.count 是方法
tricky = pd.DataFrame({"count": [1, 2], "mean": [3, 4]})
print("df['count'] 取到列 :", tricky["count"].tolist())
print("type(df.count)     :", type(tricky.count))   # 方法，不是列！
print("df.mean() 是行均值  :")
print(tricky.mean())

df['count'] 取到列 : [1, 2]
type(df.count)     : <class 'method'>
df.mean() 是行均值  :
count    1.5
mean     3.5
dtype: float64


In [15]:
# 新增列：直接赋值 / assign / insert
df["amount"] = df["sales"] * 2                     # 新增一列
df = df.assign(unit=df["sales"] / 3)              # assign 返回新表，适合链式写法
df.insert(0, "id", ["A", "B", "C", "D"])          # 插入到指定位置
df

,id,city,month,sales,amount,unit
r1,A,北京,2024-01,120.5,241.0,40.166667
r2,B,上海,2024-01,88.0,176.0,29.333333
r3,C,上海,2024-02,95.5,191.0,31.833333
r4,D,广州,2024-02,60.0,120.0,20.000000


In [16]:
# 删除列：drop 返回新表（推荐）；pop 直接改原表并返回被删的列
print(df.drop(columns=["amount"]).columns.tolist())
print("原表还有 amount 吗:", "amount" in df.columns)

popped = df.pop("unit")
print("pop 出来的 Series:", popped.tolist())
print("现在的列:", df.columns.tolist())

['id', 'city', 'month', 'sales', 'unit']
原表还有 amount 吗: True
pop 出来的 Series: [40.166666666666664, 29.333333333333332, 31.833333333333332, 20.0]
现在的列: ['id', 'city', 'month', 'sales', 'amount']


## 5. dtype：pandas 的数据类型

常见 dtype：

| dtype | 含义 | 例子 |
| --- | --- | --- |
| `int64` / `float64` | 整数 / 浮点数 | 数量、金额 |
| `bool` | 布尔 | 是否会员 |
| `str` | 字符串（**pandas 3.0 起的默认文本类型**，缺省值是 NaN） | 姓名、城市 |
| `datetime64[us]` | 时间戳 | 下单时间 |
| `category` | 有限取值集合，省内存 | 城市、渠道 |
| `object` | 万能口袋，什么都能装，但也最慢 | list、dict、混合类型 |

In [17]:
mixed = pd.DataFrame({
    "整数": [1, 2],
    "浮点": [1.5, 2.5],
    "布尔": [True, False],
    "文本": ["北京", "上海"],
    "时间": pd.to_datetime(["2024-01-01", "2024-02-01"]),
    "对象": [[1, 2], {"k": "v"}],
})
mixed.dtypes

整数             int64
浮点           float64
布尔              bool
文本               str
时间    datetime64[us]
对象            object
dtype: object

In [18]:
# 类型转换：astype 整列转 / 字典逐列指定
conv = mixed[["整数", "浮点", "文本"]].astype({"整数": "float64", "文本": "category"})
print(conv.dtypes.to_dict())
print("各列内存(字节):", conv.memory_usage(deep=True).sum())

{'整数': dtype('float64'), '浮点': dtype('float64'), '文本': CategoricalDtype(categories=['上海', '北京'], ordered=False, categories_dtype=str)}
各列内存(字节): 195


> **注意 pandas 3.0 的变化**：文本列的默认 dtype 是 `str`（以前是 `object`）。
> 它仍然能用 `pd.isna` 识别缺失，但 `astype(str)` 不会再把缺失值变成字符串 `"nan"`，而是保留缺失。
> 这一点在做数据清洗时非常关键，第 04 节会专门处理。

## 6. DataFrame 的对齐规则

- `DataFrame + Series`：按**列名**对齐（Series 的索引当成列名）
- `DataFrame + DataFrame`：按**行索引 + 列名**同时对齐

In [19]:
sales = pd.DataFrame({"北京": [100, 120], "上海": [90, 80]}, index=["1月", "2月"])
adjust = pd.Series({"北京": 10, "广州": 5})    # 只给了两个城市
print("按列对齐后相加：")
print(sales + adjust)

按列对齐后相加：
     上海   北京  广州
1月   NaN  110.0   NaN
2月   NaN  130.0   NaN


In [20]:
other = pd.DataFrame({"北京": [1, 1, 1], "上海": [1, 1, 1]}, index=["1月", "2月", "3月"])
print("行列都对齐后相加：")
print(sales + other)

行列都对齐后相加：
      北京  上海
1月  101.0  91.0
2月  121.0  81.0
3月    NaN   NaN


## 7. 视图 / 拷贝 / Copy-on-Write

pandas 3.0 默认开启 **Copy-on-Write（写时复制）**：

- 切片、筛选得到的都是"看起来像视图"的对象，但一旦修改，就会自动拷贝，原表不变。
- 因此**链式赋值 `df[条件]["列"] = 值` 永远不生效**，pandas 会给出 `ChainedAssignmentError` 警告。
- 正确写法是把"选行"和"改列"合成一步：`df.loc[条件, "列"] = 值`。

In [21]:
df = pd.DataFrame({"score": [90, 60], "level": ["A", "B"]})

df[df["score"] > 80]["level"] = "S"     # 错误示范：改的是临时副本
print("链式赋值后：", df["level"].tolist())   # 没变

df.loc[df["score"] > 80, "level"] = "S"  # 正确写法
print("loc 赋值后：", df["level"].tolist())

链式赋值后： ['A', 'B']
loc 赋值后： ['S', 'B']


C:\Users\Jason\AppData\Local\Temp\ipykernel_9996\3407058246.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  df[df["score"] > 80]["level"] = "S"     # 错误示范：改的是临时副本


In [22]:
# 真的想要一份独立副本时用 copy()，之后改副本不会影响原表
sub = df[df["score"] > 80].copy()
sub["level"] = "SS"
print("原表 :", df["level"].tolist())
print("副本 :", sub["level"].tolist())

原表 : ['S', 'B']
副本 : ['SS']


## 8. 小练习：造一张成绩表

要求：

1. 三名学生的数学 / 英语 / Python 成绩组成 DataFrame，行索引是姓名；
2. 新增 `total`（总分）、`avg`（平均分）两列；
3. 用 `pd.cut` 按平均分打等级：不及格 / 及格 / 良好 / 优秀；
4. 新增 `rank` 列表示总分排名（1 为最高）；
5. 把"赵六"的 Python 成绩改成 75（用 `.loc`）。

建议先自己写一遍，再运行下面的参考答案对比。

In [23]:
scores = pd.DataFrame(
    {
        "name": ["张三", "李四", "王五", "赵六"],
        "math": [92, 78, 85, 60],
        "english": [88, 91, 72, 65],
        "python": [95, 82, 79, 70],
    }
).set_index("name")

scores["total"] = scores.sum(axis=1)                     # axis=1 表示按行求和
scores["avg"] = scores[["math", "english", "python"]].mean(axis=1).round(1)
scores["level"] = pd.cut(
    scores["avg"],
    bins=[0, 60, 75, 90, 100],
    labels=["不及格", "及格", "良好", "优秀"],
    right=False,                                          # 左闭右开：60 算"及格"
)
scores["rank"] = scores["total"].rank(ascending=False, method="min").astype(int)

scores

,math,english,python,total,avg,level,rank
name,,,,,,,
张三,92,88,95,275,91.7,优秀,1
李四,78,91,82,251,83.7,良好,2
王五,85,72,79,236,78.7,良好,3
赵六,60,65,70,195,65.0,及格,4


In [24]:
scores.loc["赵六", "python"] = 75
scores.loc["赵六"]

math         60
english      65
python       75
total       195
avg        65.0
level      及格
rank          4
Name: 赵六, dtype: object

## 9. 小结

记住这 8 条，本章就算过关：

1. `Series` = 值 + 索引；`DataFrame` = 行索引 + 列名 + 每列 dtype。
2. 构造表最常用"字典的列表"；"字典的列表（每行 dict）"会自动补 NaN。
3. 取列用 `df["col"]`，取多列用 `df[["a", "b"]]`；少用 `df.col`。
4. 新增列直接赋值或 `assign`；`assign` 返回新表，适合链式调用。
5. `drop` 默认返回新表；`pop` 会改原表。
6. 运算默认按**标签对齐**，对不上就是 NaN——这是特性，不是 bug。
7. pandas 3.0 默认 Copy-on-Write，**链式赋值无效**，用 `.loc[条件, 列] = 值`。
8. 拿到新表先 `df.info()`：行数、列名、缺失、dtype、内存一次看清。

下一步：把磁盘上的 CSV / Excel / JSON / Parquet 读成表，见 `02_数据读写_CSV_Excel_JSON_Parquet.ipynb`。